# Context Basics

**Prerequisites:** from `01_foundations`: `03_tokens_and_context.ipynb`. From `02_agent_runtime`: `02_the_agent_loop.ipynb`. From `04_memory`: `01_what_memory_is.ipynb`

**Context is the input to one model call: everything that call is given, and nothing else.**

Every chapter so far has used the word loosely: the conversation, the prompt, what the model knows. This notebook pins it down, because the rest of the chapter makes decisions about it, and you cannot decide what goes into something you haven't defined.

The definition has a consequence that is easy to miss: **context belongs to an operation, not to a run.** One agent run makes many model calls: a `Think` deciding, an `Observe` phrasing a tool result, maybe a summarizer. Each call gets its own context, and they are not the same. Chapter 2's runtime already builds several different ones. It just never says so.

What this notebook separates:

| | What it is | Lives for |
|---|---|---|
| **conversation** | the record of what happened in a run | the run |
| **memory** | what is kept, in a list, a scratchpad or a store | as long as the store |
| **prompt** | the instructions: what to do with the information | as long as the code |
| **context** | what one call is actually sent | one call |
| **context window** | the most tokens the model can take in one call | as long as the model |

## Setup

`context.py` and `incident.py` sit one folder up. The runtime comes from `02_agent_runtime`, and `Scratchpad` from `04_memory`.

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd().parent
sys.path.append(str(HERE))
for chapter in ("01_foundations", "02_agent_runtime", "04_memory"):
    sys.path.append(str(HERE.parent / chapter))

import observe
from chat import count_tokens
from context import available, render, tokens
from incident import RULES, incident
from llm import DEFAULT_MODEL, client
from think import Think

## The session this chapter works from

`incident.py` holds one on-call session, written in the runtime's own shapes: `decision` messages for tool calls, `tool` messages holding what `Observe` said, `plan` messages, and the raw tool output in `conversation.step_outputs`. It is written out by hand rather than produced by running an agent, so that it has the things worth choosing between: rules stated once at the start, a plan that got revised, a tool called twice with the same arguments, a detour about another service, and raw results much bigger than the sentences about them.

Checkout slowed down after deploy v4.12.0 cut a database connection pool from 50 to 5. The user set two rules in their first message and never repeated them.

In [2]:
conversation, pad = incident()

for turn, m in enumerate(conversation.messages):
    what = m.get("tool") or m.get("type") or ""
    print(f"t{turn:<3}{m['role']:<10}{what:<12}{' '.join(m['content'].split())[:80]}")
print()
print(f"{len(conversation.step_outputs)} raw tool outputs in step_outputs, "
      f"{sum(count_tokens(o) for o in conversation.step_outputs)} tokens between them")
print(f"scratchpad: {pad.entries}")

t0  system                You are an on-call assistant helping investigate production incidents.
t1  user                  Checkout has been slow since this morning. Can you look into it? Two rules for t
t2  plan                  Goals: 0. Check checkout-api latency 1. Read checkout-api logs 2. Identify the c
t3  decision  get_metrics {"service": "checkout-api"}
t4  tool                  checkout-api p95 latency is 2.4 s, up from 0.3 s yesterday.
t5  decision  get_logs    {"service": "checkout-api", "since": "07:00"}
t6  tool                  The checkout-api logs since 07:00 show repeated 'connection pool exhausted' erro
t7  decision  get_deploys {"service": "checkout-api"}
t8  tool                  The last checkout-api deploy was v4.12.0 at 07:05, which changed the orders-db p
t9  assistant             Checkout p95 latency rose to 2.4 s after deploy v4.12.0 at 07:05, which cut the 
t10 user                  Unrelated, but is search-api fine? Someone mentioned it.
t11 decision  get_m

## One conversation, two contexts

Here is the claim that context belongs to an operation, checked against code that already exists. Chapter 2's `Think` and `Observe` both read this conversation, and both call a model. Nothing needs to run to see what each would send. `Think._build_messages()` returns its messages without calling anything. For `Observe`, the module's `chat` is swapped for a function that records what it was given and returns a placeholder, so no model is called.

`Observe` is handed the last `get_metrics` call as if it had just come back.

In [3]:
think_context = Think()._build_messages(conversation)

sent = {}
def capture(messages, **kwargs):
    sent["messages"] = messages
    return "(not called)"

real_chat, observe.chat = observe.chat, capture
try:
    observing, _ = incident()                     # a copy: Observe writes to step_outputs
    observing.pending = {"tool": "get_metrics", "content": observing.step_outputs[-1]}
    observe.Observe().run(observing)
finally:
    observe.chat = real_chat
observe_context = sent["messages"]

def shape(messages):
    size = sum(count_tokens(m.get("content") or "") for m in messages)
    roles = ", ".join(m["role"] for m in messages)
    return f"{len(messages)} messages, {size} tokens  [{roles}]"

print("Think  :", shape(think_context))
print("Observe:", shape(observe_context))

Think  : 23 messages, 639 tokens  [system, system, user, assistant, assistant, tool, assistant, tool, assistant, tool, assistant, user, assistant, tool, assistant, user, assistant, assistant, tool, assistant, tool, assistant, user]
Observe: 1 messages, 851 tokens  [user]


Same conversation, two different inputs.

- `Think` gets **23 messages** in the API's native shape. Each tool call is rebuilt as an assistant message carrying `tool_calls`, followed by a `tool` reply with a matching id. Chapter 2 found that models recognise finished work by that structure and not by prose. It also gets **two** system messages: the guidance `Think` writes for itself, then the session's own.
- `Observe` gets **one user message**. The whole conversation is flattened into text by `render()`, followed by the raw tool output and an instruction to state it as one sentence. It is the bigger of the two, because it's the one carrying a raw result.

Neither of these is "the conversation". Each is what one operation needed, built from it. Here is the start of what `Observe` actually sent:

In [4]:
print(observe_context[0]["content"][:1400])

Conversation so far:
system: You are an on-call assistant helping investigate production incidents.
user: Checkout has been slow since this morning. Can you look into it? Two rules for this whole session: Everything you do must be read-only: do not restart, scale or redeploy anything in production. Give every duration in milliseconds.
plan: Goals:
0. Check checkout-api latency
1. Read checkout-api logs
2. Identify the cause
decision: get_metrics({"service": "checkout-api"})
tool: checkout-api p95 latency is 2.4 s, up from 0.3 s yesterday.
decision: get_logs({"service": "checkout-api", "since": "07:00"})
tool: The checkout-api logs since 07:00 show repeated 'connection pool exhausted' errors from the orders-db client, starting at 07:12.
decision: get_deploys({"service": "checkout-api"})
tool: The last checkout-api deploy was v4.12.0 at 07:05, which changed the orders-db pool size from 50 to 5.
assistant: Checkout p95 latency rose to 2.4 s after deploy v4.12.0 at 07:05, which cut the ord

Look at the `decision:` lines. Each one is a tool call, written as `get_metrics({"service": "checkout-api"})`. When this notebook was first run they read `decision: {"service": "checkout-api"}`. A decision keeps the tool's name in a field of its own, and chapter 2's `render()` wrote only `content`, which holds the arguments. Turns 3 and 7, `get_metrics` and `get_deploys`, were identical in every prompt built from it: `Observe`, `ReflectionThink`, `PlannerThink`, all of them. The information was in the conversation the whole time; it was lost in the step that turned the conversation into a context. It was fixed in chapter 2 once it was seen here.

That's the first concrete reason to treat building a context as a job in its own right. The translation from record to input is code, it can drop things, and nothing complains when it does. It took printing what one call was actually sent to notice.

Chapter 2 made these choices one component at a time, each for its own reason. This chapter treats them as one decision, made per operation: **what does this call need, and in what form?** Chapter 2's runtime has no single place where that is decided. Each component decides for itself, inside its own `run()`. Notebook 5 comes back to that.

## Context vs conversation

**The conversation is the record. Context is what gets sent.**

The conversation holds things no model ever sees in that form. `decision` and `plan` are not API roles. A decision carries an `applied` flag that only `Act` reads. `step_outputs` isn't in `messages` at all. Every one of them has to be *translated* before it can be sent:

In [5]:
record_roles = sorted({m["role"] for m in conversation.messages})
sent_roles = sorted({m["role"] for m in think_context})
print("roles in the conversation:", record_roles)
print("roles Think sends        :", sent_roles)
print()
outputs = conversation.step_outputs
reached = [o for o in outputs if any(o in (m.get("content") or "") for m in think_context)]
print(f"raw tool outputs in the run            : {len(outputs)}, "
      f"{sum(count_tokens(o) for o in outputs)} tokens")
print(f"raw tool outputs in Think's context    : {len(reached)}")

roles in the conversation: ['assistant', 'decision', 'plan', 'system', 'tool', 'user']
roles Think sends        : ['assistant', 'system', 'tool', 'user']

raw tool outputs in the run            : 6, 1752 tokens
raw tool outputs in Think's context    : 0


About 1,750 tokens of raw output are in the run and none of them are in `Think`'s context. That was on purpose: chapter 2 decided that raw tool output never goes into `messages`, only `Observe`'s sentence about it does. That was a context decision, made before there was a word for it.

The record and the context need different properties. The record must be **complete** and must never be edited, because it's what you debug from, persist and replay. The context must be **fit for one call**, which usually means much smaller, sometimes reshaped, and different for every operation. One list can't be both, so they are two things.

## Context vs memory

**Memory is what is kept. Context is what is sent.** An item can be in memory and not in context, and then, for that call, the model doesn't know it.

The session comes with a chapter 4 `Scratchpad` holding the agent's working state: the current suspect, and what has been ruled out. It's right there in the notebook. Is it in either context?

In [6]:
suspect = pad.read("suspect")
print("scratchpad says       :", suspect)
print("in Think's context?   ", any(suspect in (m.get("content") or "") for m in think_context))
print("in Observe's context? ", any(suspect in (m.get("content") or "") for m in observe_context))

scratchpad says       : deploy v4.12.0 (orders-db pool 50 -> 5)
in Think's context?    False
in Observe's context?  False


No. Nothing reads the scratchpad, so it reaches no model call. This is chapter 4's first result again: in its second session the agent met the user as a stranger, because what it had been told was in `first.messages` and the call was built from `second.messages`. The memory existed; it just wasn't in the list being sent.

Memory is a *source* of context, one of several. Getting something from memory into a call takes a step of its own: a read, or a retrieve, followed by a decision to include it. Notebook 2 is about that decision.

## Context vs prompt

**The prompt is the instructions. Context is the information they work on.**

The words get used interchangeably, but they change at different rates. `Observe`'s message above is both at once. Split it:

In [7]:
text = observe_context[0]["content"]
cut = text.index("State this result")
information, instructions = text[:cut], text[cut:]

print(f"information : {count_tokens(information):>5} tokens -- the conversation, the raw result")
print(f"instructions: {count_tokens(instructions):>5} tokens -- 'state this result as one plain factual sentence...'")

information :   671 tokens -- the conversation, the raw result
instructions:   180 tokens -- 'state this result as one plain factual sentence...'


The instructions are the same on every `Observe` call in every run: they were written once, in `observe.py`. The information is different on every call and grows as the run goes on. Twenty turns in, it's already more than three times the size of the instructions, and on a long run it grows without limit.

This matters for more than naming:

- **Tuning a prompt and managing context are different jobs.** Every change to `Observe`'s wording in chapters 2 and 3 was a prompt change. None of them touched what information came along with it.
- **The two should be kept apart in what the model sees.** When a tool result says *"ignore previous instructions"*, what stops the model from obeying it is being able to tell data from instructions. Here they are in one string, joined only by a blank line. Notebook 4 is about keeping them apart.

## Context window

**The context window is the most tokens a model can take in one call: input and output together.** Chapter 1 hit it on purpose, and the call failed outright. Groq's model listing gives the figure for the model in use now:

In [8]:
model = next(m for m in client.models.list().data if m.id == DEFAULT_MODEL)
info = model.model_dump()
print(f"{DEFAULT_MODEL}: context window {info['context_window']:,} tokens, "
      f"at most {info['max_completion_tokens']:,} of them output")
print(f"price per million tokens: input ${float(info['pricing']['prompt']) * 1e6:.2f}, "
      f"output ${float(info['pricing']['completion']) * 1e6:.2f}")

everything = available(conversation, pad)
print(f"\neverything this session holds, as text: {tokens(everything):,} tokens")

openai/gpt-oss-120b: context window 131,072 tokens, at most 65,536 of them output
price per million tokens: input $0.15, output $0.60

everything this session holds, as text: 2,193 tokens


The whole session, raw logs included, is under 2% of the window. By that measure there's nothing to manage: send everything.

That reasoning treats the window as the limit, and it usually isn't. The window is the most the model **can** accept. How much you **should** send is a budget, and you set it per operation, for three reasons that each bind well before the window does.

## Why more context isn't necessarily better

The session's last question, answered with everything it holds. `render()` writes one line per item, labelled by kind: a stand-in for real construction, which is notebook 4.

In [9]:
price_in = float(info["pricing"]["prompt"])
price_out = float(info["pricing"]["completion"])

def ask(items):
    # `render` is a stand-in for construction: instructions as the system message, everything
    # else labelled one line per item. Notebook 4 replaces it.
    system = "\n".join(i.content for i in items if i.kind == "instruction")
    rest = [i for i in items if i.kind != "instruction"]
    response = client.chat.completions.create(model=DEFAULT_MODEL, messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": render(rest)},
    ])
    u = response.usage
    print(f"prompt {u.prompt_tokens} tokens, output {u.completion_tokens}, "
          f"cost ${u.prompt_tokens * price_in + u.completion_tokens * price_out:.5f}\n")
    print(response.choices[0].message.content)
    return response

full = ask(everything)

prompt 2347 tokens, output 532, cost $0.00067

**Root cause**  
The slowdown started right after the *checkout‑api* v4.12.0 deployment at 07:05, which reduced the **orders‑db connection pool** from 50 slots to only 5. Since 07:12 the logs show a steady rise in “connection pool exhausted” errors (5/5 in use, growing wait‑queues up to 32 waiting). Each checkout request now has to wait for a DB connection, so the service’s latency spikes and retries add extra time.

**Current impact** (last 15 min, read‑only snapshot)  
- **checkout‑api p95 latency:** 2900 ms (up from 2400 ms earlier, ≈ 2.9 × baseline ≈ 300 % increase)  
- **error rate:** 3.4 % (↑ from 2.1 %)  
- **requests/sec:** ≈ 138 (stable)  
- **CPU / memory:** unchanged (≈ 30 % CPU, ≈ 610 MiB RAM)  
- **orders‑db health:** CPU 22 %, query p95 4 ms, 41/500 connections in use – the DB itself is fine.

**What to do (read‑only actions)**  
1. **Rollback or patch the checkout‑api** to restore the orders‑db pool size to a value that can 

A good answer. It gives durations in milliseconds, as turn 1 asked, and it cites figures that only appear in the raw metrics: the 3.4% error rate, 138 requests a second. With a strong model and a couple of thousand tokens, "send everything" worked.

**The first reason is cost.** Every token of context is paid for on every call that carries it, and an agent makes many calls. Chapter 2 measured a single ReAct run at 2,466 tokens, 2,394 of them prompt. Context is most of what an agent spends, and it gets re-sent on every iteration of the loop.

**The second reason is a limit that's much closer than the window.** A session only has to run a little longer before its raw logs cover the whole morning instead of half an hour, which is an ordinary thing for a log tool to return:

In [10]:
longer, _ = incident()
lines = []
for hour in range(7, 12):
    for minute in range(60):
        lines.append(f"{hour:02d}:{minute:02d}:{(minute * 7) % 60:02d} ERROR checkout-api  "
                     f"orders-db client: connection pool exhausted (5/5 in use, {minute % 40 + 3} waiting)")
        if minute % 2 == 0:
            lines.append(f"{hour:02d}:{minute:02d}:{(minute * 11) % 60:02d} INFO  checkout-api  "
                         f"POST /checkout 200 {1800 + minute * 23}ms")
longer.step_outputs[1] = "\n".join(lines)

bigger = available(longer, pad)
print(f"{tokens(bigger):,} tokens -- {tokens(bigger) / info['context_window']:.0%} of the window\n")
try:
    ask(bigger)
except Exception as e:
    print(type(e).__name__, ":", e)

12,151 tokens -- 9% of the window



APIStatusError : Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01knxryy0kegca7rbmr648tv5x` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 13721, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Refused, at 9% of the window. The limit that applies is Groq's **tokens per minute**. On this account's tier that's 8,000 for this model, and a single request larger than that can never succeed, however long you wait. Chapter 2's `max_retries=8` doesn't help either: the SDK retries 429s, and this is a 413.

The number is specific to this provider and tier, but the pattern holds everywhere: **the limit you hit first is rarely the window.** It's a rate limit, a cost ceiling, a latency target, or the budget another part of the system needed. The window is only the last line of defence.

**The third reason is that less isn't free either, so the amount has to be chosen.** Here is the same question with only what a person skimming the session might pass on: the rules, the three sentences that carry the diagnosis, and the question.

In [11]:
at = {(i.kind, i.turn): i for i in everything}
rules = [i for i in available(conversation, constraints=RULES) if i.kind == "constraint"]
picked = [at["instruction", 0], *rules,
          at["observation", 8], at["observation", 17], at["observation", 19],
          at["request", 21]]
print(render(picked), "\n")

small = ask(picked)

[instruction] You are an on-call assistant helping investigate production incidents.
[constraint] Everything you do must be read-only: do not restart, scale or redeploy anything in production.
[constraint] Give every duration in milliseconds.
[observation] The last checkout-api deploy was v4.12.0 at 07:05, which changed the orders-db pool size from 50 to 5.
[observation] orders-db CPU is 22% and its query p95 is 4 ms; it is healthy.
[observation] checkout-api p95 latency is now 2.9 s.
[request] So what is causing the checkout slowdown, how bad is it now, and what should we do about it? A few sentences is plenty. 



prompt 226 tokens, output 325, cost $0.00023

The checkout‑api slowdown is almost certainly caused by the pool‑size change in the most recent deploy (v4.12.0 at 07:05). Reducing the orders‑db connection pool from 50 to 5 means the API now has to wait for a free DB connection far more often, which inflates request latency.

- Current checkout‑api p95 latency ≈ 2 900 ms (up from normal sub‑second values).  
- The orders‑db itself is healthy (CPU ≈ 22 % and p95 query time ≈ 4 ms), so the bottleneck is not the DB engine but the limited number of concurrent connections available to the API.

**What to do (read‑only recommendation):** Raise the orders‑db pool size back toward its original value (e.g., 40–50) and redeploy the API, then monitor the p95 latency to confirm it drops back to the expected sub‑second range. If the issue persists, check for any additional connection‑leak or timeout settings, but the pool size is the primary suspect.


A tenth of the prompt, and the answer holds up: right cause, durations in milliseconds, the database correctly ruled out. For this question most of the session was dead weight.

The cost fell by only about two thirds, though, not nine tenths. This model spends output tokens reasoning before it answers, and at four times the input price those are a large share of every call. Cutting context cuts the part of the bill it controls.

And something is gone. The error rate, 3.4%, and the 'connection pool exhausted' errors in the logs were in items that weren't picked. This answer doesn't mention either, because it can't report what it wasn't given. That's the *good* version of the failure: silence. The bad version is a model asked "how bad is it" filling the gap with a number that sounds right. Nothing in a small context tells you which of the two you'll get.

So the amount isn't the point. **For this call, the right context was a few specific items**, and knowing which ones depends on what the call is for. The *why* part of this question was answered from three sentences; the *how bad* part needed the raw metrics, and didn't get them.

## Context is chosen per operation

Put together:

- **Context is the input to one call.** Not the conversation, not the memory, not the prompt, though it's built from all three.
- **Every operation gets its own.** `Think` and `Observe` already get different contexts from the same conversation. What each one needs depends on the job it does.
- **The window is a ceiling, not a target.** What to send is a budget, and it's usually set by something other than the model: cost, rate limits, latency.
- **Neither direction is free.** Sending everything costs and eventually gets refused. Sending less can remove the one thing a question needed, and the model won't tell you it's missing.

So for each operation there's a question: *of everything available, what should this call get?* The rest of the chapter answers it in parts:

| | Question | Notebook |
|---|---|---|
| **select** | which of what is already in hand goes in? | 2 |
| **retrieve** | what should be fetched from memory that isn't in hand? | 2 |
| **compress** | what if what must go in is too big? | 3 |
| **construct** | how is what was chosen turned into the model's actual input? | 4 |
| **per component** | who decides what each of Think, Observe, ... gets? | 5 |

## What you built

Nothing new in code yet: this notebook was about definitions. But three things now exist that the rest of the chapter uses:

- **`incident()`**: one session, in the runtime's own shapes, with rules stated once, a revised plan, a repeated tool call, a detour, and raw results far larger than what was said about them.
- **`available()`**: everything a run holds as a flat list of `Item`s, each labelled with the *kind* of thing it is. A message's `role` doesn't say whether it's a request, a rule or small talk, and every later decision depends on knowing that.
- **Two risks for the next notebook to deal with:** a number that goes missing because the item holding it looked unimportant, which happened above, and a rule said once in turn 1, which anything choosing by recency will drop first.